# 01 - Auditoría inicial del dataset

Este notebook documenta la estructura, calidad y principales inconsistencias del dataset antes de cualquier transformación.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [ ]:
DATA_PATH = Path('../data/raw/sales_marketing_customer_raw.csv')
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Dimensiones y estructura

In [ ]:
print(f'Filas: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
display(df.dtypes.to_frame('dtype'))

## 2. Valores faltantes

In [ ]:
missing = pd.DataFrame({
    'faltantes': df.isna().sum(),
    'porcentaje': (df.isna().mean() * 100).round(2)
}).sort_values('porcentaje', ascending=False)
display(missing[missing['faltantes'] > 0])

## 3. Duplicados

In [ ]:
print('Duplicados completos:', df.duplicated().sum())
if 'customer_id' in df.columns:
    print('customer_id duplicados:', df['customer_id'].duplicated().sum())

## 4. Variable objetivo: churn

In [ ]:
churn_dist = df['churn'].value_counts(dropna=False).to_frame('cantidad')
churn_dist['porcentaje'] = (df['churn'].value_counts(normalize=True, dropna=False) * 100).round(2)
display(churn_dist)

## 5. Estadística descriptiva

In [ ]:
display(df.describe(include='number').T)

## 6. Controles de calidad

In [ ]:
if 'age' in df.columns:
    invalid_age = df[df['age'].notna() & ((df['age'] < 18) | (df['age'] > 100))]
    print('Edades fuera del rango 18-100:', len(invalid_age))
    display(invalid_age[['customer_id', 'age']].head(10))

In [ ]:
date_cols = ['signup_date', 'last_purchase_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

if all(col in df.columns for col in date_cols):
    temporal_errors = df[df['last_purchase_date'] < df['signup_date']]
    print('Última compra anterior al alta:', len(temporal_errors))
    display(temporal_errors[['customer_id', 'signup_date', 'last_purchase_date']].head(10))

## Próximo paso

Documentar reglas de limpieza y construir el notebook `02_data_cleaning.ipynb`.